In [ ]:
import pandas as pd
import numpy as np



In [3]:
# 1. Cargar el nuevo dataset
df = pd.read_csv('df_maestro_final.csv')

# 2. Extraer 'year' y 'week' a partir de la columna 'anio_semana'
# Separamos el string por el guión '-'
df[['year', 'week']] = df['anio_semana'].str.split('-', expand=True)
df['year'] = df['year'].astype(int)
df['week'] = df['week'].astype(int)

# 3. Crear una fecha real para el esqueleto de tiempo continuo
# El formato '%G-%V-%u' mapea Año ISO, Semana ISO y Día de la semana (1 = Lunes)
df['fecha'] = pd.to_datetime(df['year'].astype(str) + '-' + df['week'].astype(str) + '-1', format='%G-%V-%u')

# 4. Agrupar datos para evitar registros múltiples en la misma semana
df_agrupado = df.groupby(['Provincia', 'codigo_articulo', 'fecha']).agg({
    'unidades': 'sum',               # Sumamos las ventas de la misma semana
    'anio_semana': 'first',
    'tipo_abc': 'first',
    'modalidad': 'first',
    'duracion': 'max',               # Si hubo alguna promo/duración, nos quedamos con el máximo
    'hay_prueba': 'max',             # 1 si hubo prueba esa semana, 0 si no
    'clima_temp': 'mean',            # Promediamos el clima si hay varios registros
    'clima_precip': 'mean',
    'clima_viento': 'mean'
}).reset_index()

# 5. Crear el esqueleto temporal completo (sin huecos)
fechas_completas = pd.date_range(start=df['fecha'].min(), end=df['fecha'].max(), freq='W-MON')

# Obtenemos todas las combinaciones únicas de Provincia y Artículo
combinaciones = df_agrupado[['Provincia', 'codigo_articulo']].drop_duplicates()

# Producto cartesiano para tener cada artículo en cada provincia en TODAS las fechas
esqueleto = combinaciones.merge(pd.Series(fechas_completas, name='fecha'), how='cross')

# 6. Unir el esqueleto con los datos agrupados
df_completo = pd.merge(esqueleto, df_agrupado, on=['Provincia', 'codigo_articulo', 'fecha'], how='left')

# 7. Rellenar los "huecos" (Semanas sin ventas y nulos)
df_completo['unidades'] = df_completo['unidades'].fillna(0)

# Reconstruir las columnas de año y semana para las filas nuevas inventadas por el esqueleto
df_completo['year'] = df_completo['fecha'].dt.isocalendar().year
df_completo['week'] = df_completo['fecha'].dt.isocalendar().week
df_completo['anio_semana'] = df_completo['year'].astype(str) + '-' + df_completo['week'].astype(str)

# Rellenar los datos categóricos de los productos (arrastrando hacia adelante y atrás)
cols_rellenar = ['tipo_abc', 'modalidad']
df_completo[cols_rellenar] = df_completo.groupby(['codigo_articulo'])[cols_rellenar].ffill().bfill()

# Rellenar clima y otros valores con 0 cuando no haya datos previos
cols_numericas = ['duracion', 'hay_prueba', 'clima_temp', 'clima_precip', 'clima_viento']
df_completo[cols_numericas] = df_completo[cols_numericas].fillna(0)

# 8. ORDENAR CRONOLÓGICAMENTE (Paso Crítico)
df_completo = df_completo.sort_values(by=['Provincia', 'codigo_articulo', 'fecha']).reset_index(drop=True)

# 9. Calcular los Lags reales
# lag semanal = 1 semana atrás
df_completo['ventas_lag_semanal'] = df_completo.groupby(['Provincia', 'codigo_articulo'])['unidades'].shift(1).fillna(0)

# lag mensual = 4 semanas atrás
df_completo['ventas_lag_mensual'] = df_completo.groupby(['Provincia', 'codigo_articulo'])['unidades'].shift(4).fillna(0)

# 10. Organizar columnas y guardar el dataset
# Quitamos la columna 'fecha' si no la necesitas y reordenamos
df_final = df_completo[['anio_semana', 'Provincia', 'tipo_abc', 'codigo_articulo', 'unidades', 
                        'duracion', 'modalidad', 'hay_prueba', 'clima_temp', 'clima_precip', 
                        'clima_viento', 'year', 'week', 'ventas_lag_semanal', 'ventas_lag_mensual']]

# Guardar con el nombre solicitado
df_final.to_csv('df_preparado_modelado.csv', index=False)
print("¡Archivo df_pvreparado_modelado.csv generado correctamente!")

¡Archivo df_pvreparado_modelado.csv generado correctamente!
